# E2 — Masked-LM pseudo-perplexity on Colab

1. **Runtime → Change runtime type → T4 GPU.**
2. On your Mac: `python -m nlp2.make_colab_bundle` → creates `outputs/e2_colab_bundle.zip`.
3. Run the cells below in order. Everything is saved to Google Drive (`MyDrive/nlp2_e2/`), so if Colab
   disconnects just re-run all cells: already-scored segments are skipped.
4. When it finishes, download `e2_results.zip` (last cell) and unzip it into the project's `outputs/results/` folder.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
WORK = '/content/drive/MyDrive/nlp2_e2'
os.makedirs(WORK, exist_ok=True)
%cd $WORK

In [ ]:
# Upload e2_colab_bundle.zip (skipped if it is already in Drive from an earlier session)
import os, zipfile
if not os.path.exists('e2_colab_bundle.zip'):
    from google.colab import files
    files.upload()  # choose outputs/e2_colab_bundle.zip
zipfile.ZipFile('e2_colab_bundle.zip').extractall('.')
!ls nlp2 outputs/processed/conditions

In [ ]:
# Only what E2 needs (torch is preinstalled on Colab)
!pip -q install 'transformers>=4.40' sentencepiece protobuf
import torch; print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE - switch the runtime to GPU')

In [ ]:
# Smoke test: 20 segments per condition in a separate scratch folder (not mixed with the real run)
!mkdir -p /content/smoke/processed && cp -r outputs/processed/conditions /content/smoke/processed/
!NLP2_OUT_DIR=/content/smoke python -m nlp2.e2_pseudo_perplexity --limit 20

In [ ]:
# Full run: 2 models x 5 conditions x 1000 segments. Resumes from the Drive cache if interrupted.
!python -m nlp2.e2_pseudo_perplexity

In [ ]:
import pandas as pd
pd.read_csv('outputs/results/e2_pseudo_perplexity.csv').round(3)

In [ ]:
# Download results: unzip into the project's outputs/results/ on your Mac
!cd outputs/results && zip -qr /content/e2_results.zip e2 e2_pseudo_perplexity.csv
from google.colab import files
files.download('/content/e2_results.zip')

**After the gold set is tuned** (final `hist_xh_norm`): rebuild the bundle, upload it again, and re-score only that condition:

`!python -m nlp2.e2_pseudo_perplexity --conditions hist_xh_norm --overwrite`, then re-run the summary with all conditions:
`!python -m nlp2.e2_pseudo_perplexity` (cached conditions are not re-scored).